
# Black–Scholes Option Analysis Notebook (Python)

**What this does**
1. 📈 Plot a stock price chart from Yahoo Finance  
2. 📋 Retrieve all variables required for Black–Scholes modeling  
3. 🧮 Price a stock option using Black–Scholes and compare to market  
4. 🧭 Explain each variable’s role and show a pros/cons sensitivity table

> Requirements (installed automatically in the first cell):
> - yfinance
> - numpy, pandas
> - matplotlib
> - scipy


In [ ]:

# ---- Setup: install & imports ----
# If running locally and you already have these, you can skip installation lines.
%pip -q install yfinance scipy

import math
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from typing import Optional, Dict, Any, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm

# Matplotlib defaults (no specific style/colors per requirement)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True


In [ ]:

# ---- Helper utilities ----

def annualize_time_to_maturity(expiration: datetime, now: Optional[datetime] = None) -> float:
    """Compute year fraction using ACT/365 convention."""
    now = now or datetime.now(timezone.utc)
    if isinstance(expiration, str):
        expiration = datetime.fromisoformat(expiration)
    # Ensure TZ-aware
    if expiration.tzinfo is None:
        expiration = expiration.replace(tzinfo=timezone.utc)
    delta_days = (expiration - now).total_seconds() / 86400.0
    return max(delta_days, 0) / 365.0

def estimate_dividend_yield(ticker: str, lookback_years: float = 1.0) -> float:
    """Estimate forward dividend yield from last N years of dividends and current price.
    Falls back to 0.0 if no dividend data is available.
    """
    tk = yf.Ticker(ticker)
    try:
        div = tk.dividends
        if div is None or div.empty:
            return 0.0
        cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=int(365*lookback_years))
        trailing = div[div.index >= cutoff].sum()
        price = tk.fast_info.last_price if hasattr(tk, "fast_info") and tk.fast_info is not None else tk.history(period="1d")["Close"].iloc[-1]
        return float(trailing / price) if price and price > 0 else 0.0
    except Exception:
        return 0.0

def fetch_risk_free_rate(symbol: str = "^IRX") -> Optional[float]:
    """Fetch a proxy for the risk-free rate from Yahoo Finance.
    Defaults to 13-week T-bill (^IRX). Returns annualized rate in decimal.
    Fallback: None if not available.
    """
    try:
        data = yf.Ticker(symbol).history(period="5d")
        if data is None or data.empty:
            return None
        last = float(data["Close"].iloc[-1])
        # Yahoo's ^IRX is a percentage, e.g., 5.20 -> 5.20%.
        return last / 100.0
    except Exception:
        return None

def get_spot_price(ticker: str) -> float:
    tk = yf.Ticker(ticker)
    try:
        if hasattr(tk, "fast_info") and tk.fast_info is not None and tk.fast_info.last_price is not None:
            return float(tk.fast_info.last_price)
    except Exception:
        pass
    hist = tk.history(period="1d")
    if hist is None or hist.empty:
        raise ValueError("Could not fetch latest price.")
    return float(hist["Close"].iloc[-1])


In [ ]:

# ---- Black–Scholes (no arbitrage, European options) ----

@dataclass
class BSInputs:
    S: float        # Spot price
    K: float        # Strike
    r: float        # Risk-free rate (annual, cont. comp)
    q: float        # Dividend yield (annual, continuous)
    sigma: float    # Volatility (annualized)
    T: float        # Time to maturity in years

def _d1_d2(S: float, K: float, r: float, q: float, sigma: float, T: float):
    if sigma <= 0 or T <= 0 or S <= 0 or K <= 0:
        return float("nan"), float("nan")
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return d1, d2

def bs_call_price(inp: BSInputs) -> float:
    d1, d2 = _d1_d2(inp.S, inp.K, inp.r, inp.q, inp.sigma, inp.T)
    if np.isnan(d1) or np.isnan(d2):
        return float("nan")
    return inp.S * math.exp(-inp.q * inp.T) * norm.cdf(d1) - inp.K * math.exp(-inp.r * inp.T) * norm.cdf(d2)

def bs_put_price(inp: BSInputs) -> float:
    d1, d2 = _d1_d2(inp.S, inp.K, inp.r, inp.q, inp.sigma, inp.T)
    if np.isnan(d1) or np.isnan(d2):
        return float("nan")
    return inp.K * math.exp(-inp.r * inp.T) * norm.cdf(-d2) - inp.S * math.exp(-inp.q * inp.T) * norm.cdf(-d1)

def implied_volatility(target_price: float, S: float, K: float, r: float, q: float, T: float, is_call: bool = True, tol: float = 1e-7, max_iter: int = 100):
    """Bisection on volatility in [1e-6, 5.0]."""
    a, b = 1e-6, 5.0
    def price_at(vol):
        inp = BSInputs(S=S, K=K, r=r, q=q, sigma=vol, T=T)
        return bs_call_price(inp) if is_call else bs_put_price(inp)
    fa = price_at(a) - target_price
    fb = price_at(b) - target_price
    if np.isnan(fa) or np.isnan(fb):
        return None
    if fa * fb > 0:
        for b_try in [5.0, 8.0, 12.0]:
            fb = price_at(b_try) - target_price
            if fa * fb <= 0:
                b = b_try
                break
        else:
            return None
    for _ in range(max_iter):
        m = 0.5 * (a + b)
        fm = price_at(m) - target_price
        if abs(fm) < tol:
            return m
        if fa * fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    return None



## 1) Plot a stock chart from Yahoo Finance

Run the next cell to plot a configurable chart.


In [ ]:

# ---- Plot stock chart ----
ticker = "AAPL"        # <-- Change me
period = "1y"          # valid: "1mo","3mo","6mo","1y","2y","5y","10y","ytd","max"
interval = "1d"        # valid: "1d","1h","30m","15m","5m","1m" (intraday intervals limited by period)

data = yf.Ticker(ticker).history(period=period, interval=interval)
if data is None or data.empty:
    raise ValueError("No data returned. Try a different ticker/period/interval.")

plt.figure()
plt.plot(data.index, data["Close"])
plt.title(f"{ticker} Close Price ({period}, {interval})")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()



### Valuation snapshot (from Yahoo Finance)

This pulls commonly used valuation ratios for the current ticker:
- **P/E (trailing)**  
- **Price/EBITDA** (approximated as Market Cap / EBITDA)  
- **Price/Book** (P/B)


In [ ]:

# ---- Valuation metrics (P/E, Price/EBITDA, Price/Book) ----
# Re-uses the 'ticker' variable from the chart cell.
tk = yf.Ticker(ticker)

def safe_get_info(tk):
    # yfinance.get_info() returns a dict of fundamentals; keys may be missing depending on ticker
    try:
        return tk.get_info()
    except Exception:
        return {}

info = safe_get_info(tk)

# Pull fields with safe fallbacks
pe = info.get("trailingPE", None)

# Yahoo provides priceToBook; if missing, try to compute from market cap / book value
pb = info.get("priceToBook", None)

market_cap = info.get("marketCap", None)
if market_cap is None:
    try:
        market_cap = getattr(tk, "fast_info", {}).get("market_cap", None)
    except Exception:
        market_cap = None

ebitda = info.get("ebitda", None)

price_to_ebitda = None
if market_cap and ebitda and ebitda != 0:
    price_to_ebitda = float(market_cap) / float(ebitda)

# Display
print(f"Ticker: {ticker}")
print(f"P/E (trailing): {pe if pe is not None else 'N/A'}")
print(f"Price/EBITDA (Market Cap / EBITDA): {price_to_ebitda if price_to_ebitda is not None else 'N/A'}")
print(f"Price/Book (P/B): {pb if pb is not None else 'N/A'}")



## 2) Retrieve all variables needed for Black–Scholes

Fill in `ticker`, `expiration_str`, and `strike`. You can set `risk_free_override` or leave it `None` to fetch a proxy (13-week T-bill, `^IRX`).  
Dividend yield is estimated from the last 12 months of dividends.


In [ ]:

# ---- Assemble Black–Scholes variables ----
ticker = "AAPL"                             # <-- Change me
expiration_str = "2025-12-19T21:00:00+00:00" # ISO format (YYYY-MM-DD or with time & TZ)
strike = 200.0                               # <-- Change me
is_call = True                               # True for call, False for put

# Optional overrides
risk_free_override = None  # e.g., 0.045 (4.5%); set to None to fetch ^IRX
dividend_yield_override = None  # e.g., 0.005; set to None to estimate from dividends

S = get_spot_price(ticker)
T = annualize_time_to_maturity(expiration_str)

r = risk_free_override if risk_free_override is not None else (fetch_risk_free_rate("^IRX") or 0.02)
q = dividend_yield_override if dividend_yield_override is not None else estimate_dividend_yield(ticker)

# Try to fetch market option price and IV from Yahoo options (if available)
market_price = None
market_iv = None
try:
    tk = yf.Ticker(ticker)
    exps = tk.options
    if exps:
        target = pd.Timestamp(expiration_str).tz_convert(None) if pd.Timestamp(expiration_str).tzinfo else pd.Timestamp(expiration_str)
        exp_dates = pd.to_datetime(exps)
        chosen_exp = exp_dates.iloc[(exp_dates - target).abs().argmin()].strftime("%Y-%m-%d")
        chain = tk.option_chain(date=chosen_exp)
        df = chain.calls if is_call else chain.puts
        row = df.iloc[(df["strike"] - strike).abs().argmin()]
        market_price = float(row.get("lastPrice", np.nan)) if "lastPrice" in row else None
        if "impliedVolatility" in row and not pd.isna(row["impliedVolatility"]):
            market_iv = float(row["impliedVolatility"])
except Exception:
    pass

variables = {
    "S": S,
    "K": float(strike),
    "r": float(r),
    "q": float(q),
    "sigma": float(market_iv) if market_iv is not None else 0.3,  # start with market IV if available, else 30%
    "T": float(T),
    "is_call": bool(is_call),
    "market_price": float(market_price) if market_price is not None else None,
}

print("Black–Scholes input variables:")
for k, v in variables.items():
    print(f"  {k}: {v}")



## 2b) Black–Scholes variables — roles, what they affect, and sensitivities

**Model inputs**
- **$S$ (Spot price):** Current underlying price. Higher $S$ → higher call, lower put.  
  *Primary Greek:* **Delta** ($\partial V/\partial S$).  
- **$K$ (Strike):** Exercise price of the option. Higher $K$ → lower call, higher put (monotone relationship).  
  *Related sensitivity:* $-\partial C/\partial K \ge 0$, $\partial P/\partial K \ge 0$.  
- **$r$ (Risk-free rate):** Continuously compounded annual rate. Higher $r$ ↓ present value of $K$ → higher call, lower put.  
  *Primary Greek:* **Rho** ($\partial V/\partial r$; positive for calls, negative for puts).  
- **$q$ (Dividend yield):** Continuous dividend (or convenience yield). Higher $q$ lowers expected forward price → lower call, higher put.  
  *Sensitivity:* Dividend **rho** ($-\partial C/\partial q < 0$, $\partial P/\partial q > 0$).  
- **$\sigma$ (Volatility):** Annualized standard deviation of returns. Higher $\sigma$ raises both call and put (more optionality).  
  *Primary Greek:* **Vega** ($\partial V/\partial \sigma > 0$ for both).  
- **$T$ (Time to maturity in years):** More time usually increases both call and put (more time value).  
  *Primary Greek:* **Theta** (value change as time passes; typically negative as $T$ decreases).  
- **is\_call (boolean):** Chooses call vs put payoff (affects sign of many sensitivities).

> Notes: Effects assume standard conditions (European option, non-negative rates). Edge cases (very deep ITM/OTM, extreme $q$ or $r$) may show nuanced behavior.


In [ ]:

# ---- Sensitivity & "pros/cons of increasing each input" table ----
import pandas as pd

rows = [
    {
        "Variable": "S (Spot)",
        "Call effect if ↑": "↑",
        "Put effect if ↑": "↓",
        "Primary Greek": "Delta (+ for calls, − for puts)",
        "Pros of increasing (scenario)": "Stress-test bullish scenarios; shows upside convexity.",
        "Cons of increasing (scenario)": "May overstate near-term value if price move is unlikely; ignores path dependency."
    },
    {
        "Variable": "K (Strike)",
        "Call effect if ↑": "↓",
        "Put effect if ↑": "↑",
        "Primary Greek": "— (monotone w.r.t. K)",
        "Pros of increasing (scenario)": "Explores higher-strike contracts; tests sensitivity to moneyness.",
        "Cons of increasing (scenario)": "Changing K = different contract; not the same option you hold."
    },
    {
        "Variable": "r (Risk‑free)",
        "Call effect if ↑": "↑",
        "Put effect if ↑": "↓",
        "Primary Greek": "Rho (+ for calls, − for puts)",
        "Pros of increasing (scenario)": "Captures rate risk; relevant in macro shifts.",
        "Cons of increasing (scenario)": "Overstating r can underprice puts/overprice calls vs. market."
    },
    {
        "Variable": "q (Dividend yield)",
        "Call effect if ↑": "↓",
        "Put effect if ↑": "↑",
        "Primary Greek": "Dividend rho (− for calls, + for puts)",
        "Pros of increasing (scenario)": "Useful for high dividend names; forward price realism.",
        "Cons of increasing (scenario)": "Overstated q can underprice calls/overprice puts; dividend timing uncertainty."
    },
    {
        "Variable": "σ (Volatility)",
        "Call effect if ↑": "↑",
        "Put effect if ↑": "↑",
        "Primary Greek": "Vega (+ for both)",
        "Pros of increasing (scenario)": "Reflects uncertainty; captures jump/earnings risk; aligns with rich IV regimes.",
        "Cons of increasing (scenario)": "Overstated σ overprices options; sensitive to estimation window/regime shifts."
    },
    {
        "Variable": "T (Maturity)",
        "Call effect if ↑": "↑ (typical)",
        "Put effect if ↑": "↑ (typical)",
        "Primary Greek": "Theta (value falls as T ↓)",
        "Pros of increasing (scenario)": "More time value; tests long‑dated exposure/decay.",
        "Cons of increasing (scenario)": "May not match actual contract; longer T heightens model misspecification impact."
    },
]

sens_table = pd.DataFrame(rows, columns=[
    "Variable","Call effect if ↑","Put effect if ↑","Primary Greek",
    "Pros of increasing (scenario)","Cons of increasing (scenario)"
])
sens_table



## 3) Price the option using Black–Scholes

This cell computes the theoretical price and (if a market price is available) the model-implied volatility and mispricing.


In [ ]:

# ---- Black–Scholes pricing & comparison ----
inp = BSInputs(
    S=variables["S"],
    K=variables["K"],
    r=variables["r"],
    q=variables["q"],
    sigma=variables["sigma"],
    T=variables["T"],
)

theo_price = bs_call_price(inp) if variables["is_call"] else bs_put_price(inp)
print(f"Theoretical {'Call' if variables['is_call'] else 'Put'} Price (BS): {theo_price:.4f}")

if variables["market_price"] is not None:
    iv_from_market = implied_volatility(
        target_price=variables["market_price"],
        S=variables["S"],
        K=variables["K"],
        r=variables["r"],
        q=variables["q"],
        T=variables["T"],
        is_call=variables["is_call"]
    )
    print(f"""Market price: {variables['market_price']:.4f}
Implied vol from market price: {iv_from_market if iv_from_market is not None else 'N/A'}""")
    diff = variables["market_price"] - theo_price
    print(f"""Price diff (Market - BS): {diff:.4f}
{'Rich (overpriced)' if diff > 0 else 'Cheap (underpriced)' if diff < 0 else 'Fair'} relative to BS.""")
else:
    print("No market option price found for the chosen strike/expiry (or not available).")



### (Optional) Greeks

Compute common Greeks from Black–Scholes.



## 4) Greeks — what they mean and how they affect Black–Scholes modeling

**Greeks** are partial derivatives of the option value with respect to model inputs. They tell you how price changes if an input changes by a tiny amount. In Black–Scholes they have **closed‑form** expressions, which makes them fast and stable for risk and scenario analysis.

- **Delta ($\partial V/\partial S$):** Change in option price for a \$1 move in the underlying.  
  - *Calls:* $e^{-qT}N(d_1)$ (positive). *Puts:* $-e^{-qT}N(-d_1)$ (negative).  
  - *Use:* Directional exposure; delta‑hedging targets (delta‑neutral portfolios).

- **Gamma ($\partial^2 V/\partial S^2$):** Curvature—how fast **delta** changes as the price moves.  
  - Always positive for both calls and puts. Peaks near **ATM** and for **shorter maturities**.  
  - *Use:* Non‑linear risk; dictates how often you must rebalance a delta hedge.

- **Vega ($\partial V/\partial \sigma$):** Sensitivity to **volatility**; positive for both calls and puts.  
  - Larger **near ATM** and for **longer maturities**.  
  - *Use:* IV (implied vol) risk; calibration stability when backing out $\sigma$ from market prices.

- **Theta ($\partial V/\partial T$):** Sensitivity to **time to expiry**. With $T$ measured in years, theta is typically **negative** for long options (time decay).  
  - More negative near ATM and for higher $\sigma$. Short options have positive theta.  
  - *Use:* Carry/decay budgeting and roll‑down planning.

- **Rho ($\partial V/\partial r$):** Sensitivity to **interest rates**.  
  - *Calls:* positive; *Puts:* negative. Effect grows with **maturity** and **moneyness** where the discounted strike matters.  
  - *Use:* Rate risk management (more relevant for long‑dated options).

- **Dividend rho ($-\partial V/\partial q$ for calls, $+\partial V/\partial q$ for puts):** Sensitivity to **dividend yield** (or convenience yield).  
  - Higher $q$ lowers calls and lifts puts by reducing the forward price.  
  - *Use:* Important for high‑dividend equities or indices.

**Modeling implications**
- Greeks link **inputs → price moves**, guide **scenario analysis**, and drive **hedging** (delta‑/gamma‑/vega‑hedging).  
- When inferring **implied volatility**, the **vega** of the instrument controls the numerical stability of the inversion—low vega maturities/strikes lead to noisy IV.  
- **Put‑call parity** ties call/put prices and Greeks; inconsistencies can signal data quality issues.


In [ ]:

# ---- Greeks quick reference table (signs, units, and current values) ----
import pandas as pd
import math
from scipy.stats import norm

# Ensure BSInputs & _d1_d2 are available from earlier cells. If not, raise clear error.
try:
    BSInputs
    _d1_d2
except NameError as e:
    raise RuntimeError("Please run the earlier Black–Scholes setup cells first (BSInputs/_d1_d2).") from e

# Define bs_greeks here if not already defined (to allow running out of order)
try:
    bs_greeks
except NameError:
    def bs_greeks(inp: BSInputs, is_call: bool = True):
        import numpy as np
        S, K, r, q, sigma, T = inp.S, inp.K, inp.r, inp.q, inp.sigma, inp.T
        d1, d2 = _d1_d2(S, K, r, q, sigma, T)
        if np.isnan(d1) or np.isnan(d2):
            return {k: float("nan") for k in ["delta","gamma","vega","theta","rho"]}
        pdf_d1 = norm.pdf(d1)
        disc_q = math.exp(-q*T)
        disc_r = math.exp(-r*T)
        if is_call:
            delta = disc_q * norm.cdf(d1)
            theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                     - r * K * disc_r * norm.cdf(d2)
                     + q * S * disc_q * norm.cdf(d1))
            rho = K * T * disc_r * norm.cdf(d2)
        else:
            delta = -disc_q * norm.cdf(-d1)
            theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                     + r * K * disc_r * norm.cdf(-d2)
                     - q * S * disc_q * norm.cdf(-d1))
            rho = -K * T * disc_r * norm.cdf(-d2)
        gamma = disc_q * pdf_d1 / (S * sigma * math.sqrt(T))
        vega = S * disc_q * pdf_d1 * math.sqrt(T)
        return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}

# Ensure `inp` and `variables` exist; if `inp` is missing, build it from variables
try:
    variables
except NameError as e:
    raise RuntimeError("`variables` not found. Please run section 2) 'Assemble Black–Scholes variables' first.") from e

try:
    inp
except NameError:
    inp = BSInputs(S=variables["S"], K=variables["K"], r=variables["r"], q=variables["q"], sigma=variables["sigma"], T=variables["T"])

# Compute Greeks if not already present
try:
    greeks
except NameError:
    greeks = bs_greeks(inp, is_call=variables["is_call"])

is_call = variables["is_call"]

rows = [
    {
        "Greek": "Delta",
        "Sign (typical)": "Call:+  Put:−",
        "Unit": "per $1 in S",
        "Value (current)": greeks["delta"],
        "Notes": "Directional exposure; hedge ratio."
    },
    {
        "Greek": "Gamma",
        "Sign (typical)": "+ for both",
        "Unit": "per $1^2 in S",
        "Value (current)": greeks["gamma"],
        "Notes": "Curvature; re‑hedge frequency ↑ with gamma."
    },
    {
        "Greek": "Vega",
        "Sign (typical)": "+ for both",
        "Unit": "per 1.00 change in σ (i.e., 100 vol pts)",
        "Value (current)": greeks["vega"],
        "Notes": "Per 1 vol point ≈ vega × 0.01."
    },
    {
        "Greek": "Theta",
        "Sign (typical)": "Long options: −",
        "Unit": "per year of T",
        "Value (current)": greeks["theta"],
        "Notes": "Per day ≈ theta/365."
    },
    {
        "Greek": "Rho",
        "Sign (typical)": "Call:+  Put:−",
        "Unit": "per 1.00 change in r",
        "Value (current)": greeks["rho"],
        "Notes": "Rate sensitivity; larger for long‑dated."
    },
]

df_quick = pd.DataFrame(rows, columns=["Greek","Sign (typical)","Unit","Value (current)","Notes"])
df_quick


In [ ]:

# ---- Finite-difference sanity check (compares analytic Greeks vs. bumps) ----
import numpy as np
import pandas as pd

# Ensure required objects exist
try:
    BSInputs
except NameError as e:
    raise RuntimeError("Please run the Black–Scholes setup cells first.") from e

try:
    inp
except NameError:
    inp = BSInputs(S=variables["S"], K=variables["K"], r=variables["r"], q=variables["q"], sigma=variables["sigma"], T=variables["T"])

# Define bs_greeks if missing and compute greeks
try:
    bs_greeks
except NameError:
    import math
    from scipy.stats import norm
    def bs_greeks(inp: BSInputs, is_call: bool = True):
        S, K, r, q, sigma, T = inp.S, inp.K, inp.r, inp.q, inp.sigma, inp.T
        d1, d2 = _d1_d2(S, K, r, q, sigma, T)
        if np.isnan(d1) or np.isnan(d2):
            return {k: float("nan") for k in ["delta","gamma","vega","theta","rho"]}
        pdf_d1 = norm.pdf(d1)
        disc_q = math.exp(-q*T)
        disc_r = math.exp(-r*T)
        if is_call:
            delta = disc_q * norm.cdf(d1)
            theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                     - r * K * disc_r * norm.cdf(d2)
                     + q * S * disc_q * norm.cdf(d1))
            rho = K * T * disc_r * norm.cdf(d2)
        else:
            delta = -disc_q * norm.cdf(-d1)
            theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                     + r * K * disc_r * norm.cdf(-d2)
                     - q * S * disc_q * norm.cdf(-d1))
            rho = -K * T * disc_r * norm.cdf(-d2)
        gamma = disc_q * pdf_d1 / (S * sigma * math.sqrt(T))
        vega = S * disc_q * pdf_d1 * math.sqrt(T)
        return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}

try:
    greeks
except NameError:
    greeks = bs_greeks(inp, is_call=variables["is_call"])

def price_option(inp, is_call=True):
    return bs_call_price(inp) if is_call else bs_put_price(inp)

base_price = price_option(inp, is_call=variables["is_call"])

# Bump sizes
dS = max(0.005 * inp.S, 0.01)          # 0.5% spot or 1 cent minimum
dSigma = 0.01                          # +1 vol point
dT = min(1/365, max(1e-6, 0.1*inp.T))  # ~1 day (yrs) or smaller
dr = 1e-4                               # +1 bp
dq = 1e-4                               # +1 bp

# Delta (1st derivative wrt S)
inp_up = BSInputs(S=inp.S + dS, K=inp.K, r=inp.r, q=inp.q, sigma=inp.sigma, T=inp.T)
inp_dn = BSInputs(S=max(1e-8, inp.S - dS), K=inp.K, r=inp.r, q=inp.q, sigma=inp.sigma, T=inp.T)
delta_fd = (price_option(inp_up, variables["is_call"]) - price_option(inp_dn, variables["is_call"])) / (2*dS)

# Gamma (2nd derivative wrt S)
gamma_fd = (price_option(inp_up, variables["is_call"]) - 2*base_price + price_option(inp_dn, variables["is_call"])) / (dS**2)

# Vega (derivative wrt sigma)
inp_vs = BSInputs(S=inp.S, K=inp.K, r=inp.r, q=inp.q, sigma=inp.sigma + dSigma, T=inp.T)
vega_fd = (price_option(inp_vs, variables["is_call"]) - base_price) / dSigma

# Theta (derivative wrt T)
T_new = max(1e-6, inp.T - dT)
inp_th = BSInputs(S=inp.S, K=inp.K, r=inp.r, q=inp.q, sigma=inp.sigma, T=T_new)
theta_fd = (price_option(inp_th, variables["is_call"]) - base_price) / (T_new - inp.T)

# Rho (derivative wrt r)
inp_r = BSInputs(S=inp.S, K=inp.K, r=inp.r + dr, q=inp.q, sigma=inp.sigma, T=inp.T)
rho_fd = (price_option(inp_r, variables["is_call"]) - base_price) / dr

# Dividend rho (derivative wrt q)
inp_q = BSInputs(S=inp.S, K=inp.K, r=inp.r, q=inp.q + dq, sigma=inp.sigma, T=inp.T)
divrho_fd = (price_option(inp_q, variables["is_call"]) - base_price) / dq

comp = pd.DataFrame({
    "Analytic": [greeks["delta"], greeks["gamma"], greeks["vega"], greeks["theta"], greeks["rho"]],
    "Finite‑diff": [delta_fd, gamma_fd, vega_fd, theta_fd, rho_fd],
    "Units": ["per $1 S", "per $1^2 S", "per 1.00 σ", "per 1.00 T (yrs)", "per 1.00 r"],
}, index=["Delta","Gamma","Vega","Theta","Rho"])
comp


In [ ]:

def bs_greeks(inp: BSInputs, is_call: bool = True) -> Dict[str, float]:
    S, K, r, q, sigma, T = inp.S, inp.K, inp.r, inp.q, inp.sigma, inp.T
    d1, d2 = _d1_d2(S, K, r, q, sigma, T)
    if np.isnan(d1) or np.isnan(d2):
        return {k: float("nan") for k in ["delta","gamma","vega","theta","rho"]}
    pdf_d1 = norm.pdf(d1)
    disc_q = math.exp(-q*T)
    disc_r = math.exp(-r*T)
    if is_call:
        delta = disc_q * norm.cdf(d1)
        theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                 - r * K * disc_r * norm.cdf(d2)
                 + q * S * disc_q * norm.cdf(d1))
        rho = K * T * disc_r * norm.cdf(d2)
    else:
        delta = -disc_q * norm.cdf(-d1)
        theta = (- (S * disc_q * pdf_d1 * sigma) / (2 * math.sqrt(T))
                 + r * K * disc_r * norm.cdf(-d2)
                 - q * S * disc_q * norm.cdf(-d1))
        rho = -K * T * disc_r * norm.cdf(-d2)
    gamma = disc_q * pdf_d1 / (S * sigma * math.sqrt(T))
    vega = S * disc_q * pdf_d1 * math.sqrt(T)
    return {"delta": delta, "gamma": gamma, "vega": vega, "theta": theta, "rho": rho}

greeks = bs_greeks(inp, is_call=variables["is_call"])
greeks
